# AI Model Validation Notebook

Validates YOLOv8n, CLIP, rembg, and Real-ESRGAN models.

In [ ]:
# Install dependencies
!pip install -q rembg ultralytics open_clip_torch realesrgan pynvml pillow numpy pandas

In [ ]:
import os
import time
import json
from pathlib import Path
from PIL import Image
import pandas as pd
import torch

# Categories to validate
CATEGORIES = ['perfume', 'cosmetics', 'furniture', 'electronics', 'food', 'shoes', 'fashion']
IMAGES_PER_CATEGORY = 20

# Create synthetic test images
def create_test_image(category: str) -> Image.Image:
    img = Image.new('RGB', (1024, 1024), (245, 245, 240))
    from PIL import ImageDraw
    draw = ImageDraw.Draw(img)
    draw.rectangle([200, 200, 800, 800], fill=(200, 150, 100))
    return img

## 1. Background Removal (rembg) Validation

In [ ]:
from rembg import remove

def benchmark_rembg(image: Image.Image):
    start = time.time()
    result = remove(image)
    duration = time.time() - start
    return {'duration': duration, 'output_size': len(result)}

## 2. YOLOv8n Detection Validation

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolov8n.pt')

def benchmark_yolo(image: Image.Image):
    start = time.time()
    results = yolo(image, verbose=False)
    duration = time.time() - start
    boxes = results[0].boxes
    return {
        'duration': duration,
        'detections': len(boxes) if boxes else 0,
        'confidence': float(boxes[0].conf[0]) if boxes else 0.0
    }

## 3. CLIP Classification Validation

In [ ]:
import open_clip

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')

CATEGORY_LABELS = CATEGORIES

def benchmark_clip(image: Image.Image):
    start = time.time()
    inputs = preprocess(image).unsqueeze(0)
    text = tokenizer(CATEGORY_LABELS)
    with torch.no_grad():
        image_features = model.encode_image(inputs)
        text_features = model.encode_text(text)
        probs = (image_features @ text_features.T).softmax(dim=-1)
    pred_idx = probs.argmax().item()
    duration = time.time() - start
    return {
        'duration': duration,
        'predicted_category': CATEGORY_LABELS[pred_idx],
        'confidence': float(probs[0][pred_idx])
    }

## 4. Real-ESRGAN Enhancement Validation

In [ ]:
from realesrgan import RealESRGAN

enhancer = RealESRGAN('cuda', scale=2)

def benchmark_esrgan(image: Image.Image):
    start = time.time()
sr = enhancer.predict(image)
    duration = time.time() - start
    return {'duration': duration, 'output_size': sr.size}

## 5. Full Validation Loop

In [ ]:
results = []
for category in CATEGORIES:
    for i in range(IMAGES_PER_CATEGORY):
        img = create_test_image(category)
        row = {'category': category, 'image': i}
        
        row.update({'rembg_' + k: v for k, v in benchmark_rembg(img).items()})
        row.update({'yolo_' + k: v for k, v in benchmark_yolo(img).items()})
        row.update({'clip_' + k: v for k, v in benchmark_clip(img).items()})
        row.update({'esrgan_' + k: v for k, v in benchmark_esrgan(img).items()})
        
        results.append(row)
        print(f'{category} - image {i+1}/{IMAGES_PER_CATEGORY}')

df = pd.DataFrame(results)
df.to_csv('validation_results.csv', index=False)

## 6. GPU Memory Monitoring

In [ ]:
import pynvml
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
print(f'VRAM Used: {mem.used / 1024**3:.2f} GB')
print(f'VRAM Total: {mem.total / 1024**3:.2f} GB')